# E791 Laura++ genfit ensemble

This notebook generates and fits $N$ independent pseudoexperiments with the seven-component E791 $D^+\to\pi^-\pi^+\pi^+$ Fit 2 model. Every fit uses the Laura++-style Gauss--Legendre normalization. The ensemble tests whether the mean fitted central value is compatible with the generated value and whether the parameter pulls have mean zero and width one.

The $\rho(770)$ coefficient is fixed to $1+0i$. The other six complex coefficients give twelve free Cartesian parameters.

In [ ]:
import time

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from dalitzplotfitter import (
    DecayChannel, DecayModel, Minimizer, NonResonant, Parameter,
    RealImag, Resonance, enable_x64, weighted_resample,
)

enable_x64()

## Ensemble configuration

Increase `N_EXPERIMENTS` further for the final bias study. One hundred toys provide a useful first compatibility test, but a few hundred or more are normally needed for precise pull-width and small-bias measurements.

In [ ]:
N_EXPERIMENTS = 100
EVENTS_PER_EXPERIMENT = 10_000
POOL_SIZE = 3_000_000
POOL_SEED = 2000
TOY_SEED = 791
START_SEED = 314159
START_SIGMA = 0.12
MAX_CALLS = 50_000
LAURA_BIN_WIDTH = 0.005

## E791 Fit 2 model and generated parameters

In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
fit2_polar = {
    "sigma": (1.17, 205.7), "rho770": (1.00, 0.0),
    "NR": (0.48, 57.3), "f0_980": (0.43, 165.0),
    "f2_1270": (0.76, 57.3), "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def internal_xy(name):
    magnitude, phase_deg = fit2_polar[name]
    if name == "NR":
        phase_deg += 180.0
    phase = np.deg2rad(phase_deg)
    return magnitude * np.cos(phase), magnitude * np.sin(phase)

truth = {}

def free_coefficient(name):
    x_truth, y_truth = internal_xy(name)
    truth[f"{name}.x"] = float(x_truth)
    truth[f"{name}.y"] = float(y_truth)
    return RealImag(
        Parameter.coefficient(f"{name}.x", x_truth, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", y_truth, owner=name, step=0.01),
    )

coefficients = {
    "sigma": free_coefficient("sigma"),
    "rho770": RealImag(1.0, 0.0),
    "NR": free_coefficient("NR"),
    "f0_980": free_coefficient("f0_980"),
    "f2_1270": free_coefficient("f2_1270"),
    "f0_1370": free_coefficient("f0_1370"),
    "rho1450": free_coefficient("rho1450"),
}
components = [
    Resonance("sigma", (0, 1), coefficients["sigma"], mass=0.478, width=0.324, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770", (0, 1), coefficients["rho770"], mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980", (0, 1), coefficients["f0_980"], mass=0.975, width=0.044, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0, 1), coefficients["f2_1270"], mass=1.275, width=0.185, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0, 1), coefficients["f0_1370"], mass=1.434, width=0.173, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0, 1), coefficients["rho1450"], mass=1.465, width=0.310, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]
model = DecayModel(
    channel, components, normalize_components=True,
    normalization_method="laura",
    normalization_bin_width=LAURA_BIN_WIDTH,
)
parameter_names = [parameter.name for parameter in model.parameters if not parameter.fixed]
truth_vector = np.asarray([truth[name] for name in parameter_names])
print(f"parameters: {len(parameter_names)}")
print(f"Laura normalization points: {model.normalization_sample.size:,}")

## Shared generation pool

The large weighted phase-space pool is constructed once. Each pseudoexperiment uses an independent resampling key, so the event samples fluctuate independently conditional on this numerical proposal pool.

In [ ]:
pool = model.generate_phase_space(POOL_SIZE, seed=POOL_SEED)
truth_intensity = model.intensity(pool.as_dict(), truth)
truth_normalization = jnp.mean(
    model.normalization_sample.weights
    * model.intensity(model.normalization_sample.as_dict(), truth)
)
generation_weights = pool.weights * truth_intensity
assert bool(jnp.all(jnp.isfinite(generation_weights)))
assert float(jnp.sum(generation_weights)) > 0.0
print(f"pool events: {pool.size:,}")
print(f"truth normalization: {float(truth_normalization):.12g}")

## Generate and fit the ensemble

In [ ]:
fit_values = np.full((N_EXPERIMENTS, len(parameter_names)), np.nan)
fit_errors = np.full_like(fit_values, np.nan)
fit_valid = np.zeros(N_EXPERIMENTS, dtype=bool)
fit_edm = np.full(N_EXPERIMENTS, np.nan)
fit_nfcn = np.zeros(N_EXPERIMENTS, dtype=int)
fit_delta_nll = np.full(N_EXPERIMENTS, np.nan)
start_rng = np.random.default_rng(START_SEED)
start_time = time.perf_counter()

for toy_index in range(N_EXPERIMENTS):
    data = weighted_resample(
        jax.random.key(TOY_SEED + toy_index),
        pool, generation_weights, EVENTS_PER_EXPERIMENT, replace=True,
    )
    cache = model.prepare_cache(data)

    def nll(values):
        intensity, normalization = cache.evaluate(values)
        return (
            -jnp.sum(jnp.log(jnp.clip(intensity, min=1.0e-300)))
            + data.size * jnp.log(normalization)
        )

    start = {
        name: float(truth[name] + start_rng.normal(0.0, START_SIGMA))
        for name in parameter_names
    }
    minimizer = Minimizer(nll, model.parameters, tolerance=1.0e-4, verbose=0)
    result = minimizer.fit(
        start_values=start, simplex=False, ncall=MAX_CALLS
    )
    fit_valid[toy_index] = bool(result.valid)
    fit_edm[toy_index] = float(result.fmin.edm)
    fit_nfcn[toy_index] = int(result.nfcn)
    fit_delta_nll[toy_index] = float(result.fval - nll(truth))
    for parameter_index, name in enumerate(parameter_names):
        fit_values[toy_index, parameter_index] = float(result.values[name])
        fit_errors[toy_index, parameter_index] = float(result.errors[name])
    print(
        f"toy {toy_index + 1:3d}/{N_EXPERIMENTS}: "
        f"valid={fit_valid[toy_index]} "
        f"EDM={fit_edm[toy_index]:.2e} "
        f"DeltaNLL={fit_delta_nll[toy_index]:+.3f}"
    )

elapsed = time.perf_counter() - start_time
print(f"elapsed: {elapsed:.1f} s ({elapsed / N_EXPERIMENTS:.2f} s/toy)")

## Ensemble summary

For each parameter, the compatibility statistic is

$$z_{bias}=\frac{\langle\hat\theta\rangle-\theta_{true}}{s(\hat\theta)/\sqrt{N_{valid}}}.$$

It tests the fitted ensemble mean against the generated value. The pull is $(\hat\theta-\theta_{true})/\sigma_{fit}$.

In [ ]:
usable = (
    fit_valid
    & np.all(np.isfinite(fit_values), axis=1)
    & np.all(np.isfinite(fit_errors) & (fit_errors > 0.0), axis=1)
)
n_valid = int(np.sum(usable))
values = fit_values[usable]
errors = fit_errors[usable]
pulls = (values - truth_vector[None, :]) / errors
means = np.mean(values, axis=0)
spread = np.std(values, axis=0, ddof=1)
mean_errors = spread / np.sqrt(n_valid)
bias = means - truth_vector
bias_z = bias / mean_errors
reported_error = np.mean(errors, axis=0)
pull_mean = np.mean(pulls, axis=0)
pull_width = np.std(pulls, axis=0, ddof=1)

print(f"valid fits: {n_valid}/{N_EXPERIMENTS} ({n_valid / N_EXPERIMENTS:.1%})")
print(f"{'parameter':16s} {'truth':>9s} {'mean fit':>10s} {'SEM':>9s} {'bias':>9s} {'z_bias':>8s} {'mean err':>9s} {'pull mu':>8s} {'pull sig':>9s}")
for index, name in enumerate(parameter_names):
    print(
        f"{name:16s} {truth_vector[index]:9.4f} {means[index]:10.4f} "
        f"{mean_errors[index]:9.4f} {bias[index]:+9.4f} {bias_z[index]:+8.2f} "
        f"{reported_error[index]:9.4f} {pull_mean[index]:+8.2f} {pull_width[index]:9.2f}"
    )

assert n_valid > 1

In [ ]:
positions = np.arange(len(parameter_names))
fig, axes = plt.subplots(2, 1, figsize=(12, 9), constrained_layout=True)
axes[0].errorbar(positions, means, yerr=mean_errors, fmt="o", capsize=3, label="ensemble mean +/- SEM")
axes[0].scatter(positions, truth_vector, marker="x", s=70, label="generated")
axes[0].set_ylabel("coefficient value")
axes[0].set_title("Mean fitted central values versus generated values")
axes[0].legend()
axes[1].axhspan(-2.0, 2.0, alpha=0.12, color="tab:green")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].scatter(positions, bias_z)
axes[1].set_ylabel(r"$z_{bias}$")
axes[1].set_title("Compatibility of ensemble mean with generated value")
for axis in axes:
    axis.set_xticks(positions, parameter_names, rotation=45, ha="right")
plt.show()

In [ ]:
x_gaussian = np.linspace(-4.0, 4.0, 400)
standard_normal = np.exp(-0.5 * x_gaussian**2) / np.sqrt(2.0 * np.pi)
fig, axes = plt.subplots(3, 4, figsize=(15, 10), constrained_layout=True)
for index, (axis, name) in enumerate(zip(axes.flat, parameter_names)):
    axis.hist(pulls[:, index], bins=12, range=(-4, 4), density=True, alpha=0.65)
    axis.plot(x_gaussian, standard_normal, color="black", linewidth=1.2)
    axis.set_title(f"{name}\nmu={pull_mean[index]:+.2f}, sigma={pull_width[index]:.2f}")
    axis.set_xlabel("pull")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
axes[0].hist(fit_edm[np.isfinite(fit_edm)], bins=15)
axes[0].set(xlabel="EDM", title="Fit convergence")
axes[1].hist(fit_nfcn, bins=15)
axes[1].set(xlabel="function calls", title="Minimizer cost")
axes[2].hist(fit_delta_nll[np.isfinite(fit_delta_nll)], bins=15)
axes[2].set(xlabel=r"$NLL_{fit}-NLL_{truth}$", title="Statistical improvement over truth")
plt.show()

## Interpretation

A satisfactory ensemble should have a high valid-fit fraction, fitted means statistically compatible with the generated values, pull means compatible with zero, and pull widths compatible with one. One hundred toys still leave visible statistical uncertainty, especially when inspecting twelve parameters simultaneously. Increase `N_EXPERIMENTS` before drawing a final conclusion about small bias or coverage. Repeat the ensemble at finer Laura++ integration orders to separate minimizer/statistical effects from quadrature bias.